# Agentic RAG: Router-Retriever System with PDF and Web Search Tools

**Course-End Project**

This notebook implements a multi-agent Agentic RAG system using CrewAI:
- A **Router Agent** classifies each question and decides which knowledge source(s) to use: the PDF knowledge base, live web search, the LLM's own knowledge, or a combination of these.
- A **Retriever Agent** executes the retrieval method(s) chosen by the Router and produces a grounded, synthesized answer.

**LLM:** Google Gemini (via CrewAI's LiteLLM integration)  
**Web search:** Tavily  
**PDF search:** FAISS vector store over `pdf4.pdf` (Payet & Prince, *Maximizing Profits with Five Indispensable CX Practices*, Part 4)


## Step 1: Install required packages

In [ ]:
!pip install -U crewai langchain langchain-community langchain-classic langchain-text-splitters langchain-google-genai faiss-cpu tiktoken pypdf tavily-python python-dotenv litellm


## Step 2: Imports

In [ ]:
import os
import json
import datetime
import requests

from dotenv import load_dotenv

from crewai import Agent, Task, Crew
from crewai.llm import LLM
from crewai.tools import BaseTool

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings


## Step 3: Load credentials

**Do not hardcode API keys in this notebook.** Create a `.env` file in the same folder as this notebook with the following contents, then load it below:

```
GOOGLE_API_KEY=your_google_api_key_here
TAVILY_API_KEY=your_tavily_api_key_here
```


In [ ]:
load_dotenv()

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
TAVILY_API_KEY = os.environ["TAVILY_API_KEY"]


## Step 4: Set up the Gemini LLM (used by both agents)

In [ ]:
llm = LLM(
    model="gemini/gemini-2.5-flash",  # confirm this matches a model your API key has access to
    api_key=GOOGLE_API_KEY,
    temperature=0
)


## Step 5: Build the PDF knowledge base (FAISS vector store)

Loads, chunks, and embeds `pdf4.pdf`. Make sure the file is in the same directory as this notebook.

In [ ]:
pdf_paths = ["pdf4.pdf"]

documents = []
for path in pdf_paths:
    documents.extend(PyPDFLoader(path).load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
    task_type="retrieval_document"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

print(f"Loaded {len(pdf_paths)} PDF(s) into {len(docs)} chunks")


## Step 6: Define the tools

Two custom CrewAI tools: one for PDF search (FAISS), one for live web search (Tavily).

In [ ]:
class PDFSearchTool(BaseTool):
    name: str = "pdf_search"
    description: str = (
        "Search the internal PDF knowledge base for domain-specific information about "
        "customer experience (CX) strategy, service protocols, onboarding, workforce management, "
        "and quality assurance practices."
    )

    def _run(self, query: str):
        results = retriever.invoke(query)
        if not results:
            return "No relevant PDF content found."
        return "\n\n".join(d.page_content for d in results)


class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent, current, or time-sensitive information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }
        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title} - {link}")

        return "\n".join(results) if results else "No web results found."


pdf_tool = PDFSearchTool()
search_tool = TavilySearchTool()


## Step 7: Define the agents

**Router Agent** — classifies the question and decides which source(s) to use. It does not retrieve anything itself.

**Retriever Agent** — has both tools attached, executes the source(s) chosen by the Router, and synthesizes the final answer.

In [ ]:
router = Agent(
    role="Router",
    goal=(
        "Decide which knowledge source(s) are needed to answer the question: the PDF knowledge base, "
        "live web search, the LLM's own knowledge, or a combination."
    ),
    backstory=(
        "You are an expert at analyzing questions and identifying where the best answer will come from. "
        "The PDF knowledge base contains an article on customer experience (CX) strategy, specifically covering: "
        "establishing customer service protocols and contact-reason documentation, onboarding and continuous "
        "improvement practices for CX teams, workforce management (scheduling, forecasting, staffing), and "
        "quality assurance / service optimization practices. "
        "Web search should be used for time-sensitive, current, or recent information not covered by the PDF. "
        "The LLM's own general knowledge should be used for questions that do not need retrieval at all. "
        "Some questions genuinely need more than one source, e.g. a question partly covered by the PDF "
        "that also benefits from current context."
    ),
    llm=llm,
    verbose=True
)

retriever_agent = Agent(
    role="Retriever",
    goal="Use the source(s) selected by the Router to gather information and produce one accurate, synthesized answer.",
    backstory=(
        "You answer questions by pulling from whichever combination of sources the Router indicates. "
        "You always check whether retrieved content actually answers the question before relying on it, "
        "rather than forcing an answer from weak or irrelevant material."
    ),
    llm=llm,
    tools=[pdf_tool, search_tool],
    verbose=True
)


## Step 8: Define a reusable function to run the crew on any question

Builds fresh Router/Retriever tasks per question (so `context` wiring is correct each run), executes the crew, and returns both the result and a reasoning trace.

In [ ]:
def run_agentic_rag(user_question: str):
    task_route = Task(
        description=(
            f"Analyze this question: '{user_question}'\n\n"
            "Decide which knowledge source(s) are needed. Choose one or more of: 'pdf', 'web', 'direct'.\n"
            "- 'pdf' -> question is about our internal/domain content covered in the PDF knowledge base\n"
            "- 'web' -> question needs current, real-time, or recent information\n"
            "- 'direct' -> general knowledge answerable without retrieval\n"
            "- Combination (e.g. 'pdf, web') -> the question genuinely needs more than one source\n\n"
            "Output ONLY a comma-separated list of the chosen source(s), followed by a one-sentence justification."
        ),
        expected_output="A comma-separated routing decision (e.g. 'pdf' or 'pdf, web') with brief justification.",
        agent=router
    )

    task_retrieve = Task(
        description=(
            f"The Router has determined which source(s) to use for this question: '{user_question}'.\n\n"
            "- If 'pdf' is indicated, call the pdf_search tool.\n"
            "- If 'web' is indicated, call the web_search tool.\n"
            "- If 'direct' is indicated, answer from your own knowledge without calling a tool.\n"
            "- If multiple sources are indicated, use all of them and synthesize one coherent answer, "
            "noting which parts came from which source.\n\n"
            "IMPORTANT: If the retrieved PDF content doesn't actually answer the question, say so explicitly "
            "rather than guessing or forcing an answer from irrelevant chunks.\n\n"
            "Produce a final, grounded answer to the question."
        ),
        expected_output="A synthesized, source-attributed answer to the question.",
        agent=retriever_agent,
        context=[task_route]
    )

    crew = Crew(
        agents=[router, retriever_agent],
        tasks=[task_route, task_retrieve],
        verbose=True
    )

    result = crew.kickoff()

    trace = []
    for task in [task_route, task_retrieve]:
        trace.append({
            "agent": task.agent.role,
            "output": str(task.output.raw) if task.output else None,
            "timestamp": datetime.datetime.now().isoformat()
        })

    return result, trace


## Step 9: Test questions

Four questions designed to exercise each routing path: PDF-only, web-only, direct, and a combination.

In [ ]:
test_questions = [
    "What are the four practices for implementing operational CX processes?",  # expect: pdf
    "What are the latest customer experience trends in 2026?",  # expect: web
    "What does CX stand for?",  # expect: direct
    "How do current AI trends in customer support relate to the QA practices described in the article?",  # expect: pdf, web
]

all_traces = []

for q in test_questions:
    print("=" * 80)
    print(f"QUESTION: {q}")
    print("=" * 80)

    result, trace = run_agentic_rag(q)

    print("\n--- FINAL ANSWER ---")
    print(result.raw)
    print()

    all_traces.append({"question": q, "trace": trace})


## Step 10: Save reasoning trace log

Writes all agent decisions and outputs to `reasoning_trace.json` for traceability, as required by the project.

In [ ]:
with open("reasoning_trace.json", "w") as f:
    json.dump(all_traces, f, indent=2)

print("Saved reasoning_trace.json")
print(json.dumps(all_traces, indent=2))


## Step 11: (Optional) Ask a custom question interactively

In [ ]:
my_question = "How should a SaaS company approach onboarding new CX employees?"
result, trace = run_agentic_rag(my_question)

print("--- FINAL ANSWER ---")
print(result.raw)

print("\n--- REASONING TRACE ---")
print(json.dumps(trace, indent=2))
